In [0]:
from pyspark.sql.functions import (
    current_timestamp, lit, col, to_date,
    sum as spark_sum, current_date, to_timestamp
)
from pyspark.sql.window import Window

VOLUME_PATH = "/Volumes/de_workspace26/ecommerce_pawan/raw_files"
CATALOG     = "de_workspace26"
SCHEMA_B    = f"{CATALOG}.bronze_pawan"
SCHEMA_S    = f"{CATALOG}.silver_pawan"
SCHEMA_G    = f"{CATALOG}.gold_pawan"

print("Constants set.")
print("Volume path :", VOLUME_PATH)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {SCHEMA_G}.live_orders")

dbutils.fs.rm(f"{VOLUME_PATH}/_checkpoints/live_orders", recurse=True)
dbutils.fs.rm(f"{VOLUME_PATH}/_schema/live_orders",      recurse=True)

print("✅ Dropped live_orders table.")
print("✅ Checkpoint and schema cleared.")

In [0]:

ORDERS_RAW_PATH = f"{VOLUME_PATH}/orders_raw"
files = dbutils.fs.ls(ORDERS_RAW_PATH)
print(f"Files in orders_raw: {len(files)}")
for f in files:
    print(f"  - {f.name}  ({f.size} bytes)")

assert len(files) == 2, f"❌ Expected 2 files but found {len(files)}"
print("\n✅ Source files confirmed:")
print("   orders.csv        → 205 rows")
print("   orders_batch2.csv → 30  rows")
print("   Expected total    → 235 rows")

In [0]:
# Pre-create directories in volume — same pattern that worked for bronze
CHECKPOINT_LIVE = f"{VOLUME_PATH}/_checkpoints/live_orders"
SCHEMA_LOC_LIVE = f"{VOLUME_PATH}/_schema/live_orders"

dbutils.fs.mkdirs(CHECKPOINT_LIVE)
dbutils.fs.mkdirs(SCHEMA_LOC_LIVE)

# Verify dirs exist
try:
    dbutils.fs.ls(CHECKPOINT_LIVE)
    print(f"✅ Checkpoint dir exists : {CHECKPOINT_LIVE}")
except:
    print(f"❌ Checkpoint dir missing : {CHECKPOINT_LIVE}")

try:
    dbutils.fs.ls(SCHEMA_LOC_LIVE)
    print(f"✅ Schema dir exists     : {SCHEMA_LOC_LIVE}")
except:
    print(f"❌ Schema dir missing    : {SCHEMA_LOC_LIVE}")

In [0]:
print("=== Checkpoint files ===")
try:
    ck_files = dbutils.fs.ls(CHECKPOINT_LIVE)
    for f in ck_files:
        print(f"  - {f.name}")
    print(f"✅ Checkpoint created at : {CHECKPOINT_LIVE}")
except Exception as e:
    print(f"❌ Checkpoint not found  : {e}")

print("\n=== Schema files ===")
try:
    sc_files = dbutils.fs.ls(SCHEMA_LOC_LIVE)
    for f in sc_files:
        print(f"  - {f.name}")
    print(f"✅ Schema created at     : {SCHEMA_LOC_LIVE}")
except Exception as e:
    print(f"❌ Schema not found      : {e}")

In [0]:
stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC_LIVE)
        .load(ORDERS_RAW_PATH)
)

print("✅ Streaming DataFrame configured.")
print(f"   Source          : {ORDERS_RAW_PATH}")
print(f"   Schema location : {SCHEMA_LOC_LIVE}")
print(f"   isStreaming     : {stream_df.isStreaming}")   # True

In [0]:
stream_wm = (
    stream_df
        .withColumn(
            "order_date",
            to_timestamp(col("order_date"), "yyyy-MM-dd")
        )
        .withWatermark("order_date", "10 minutes")
)

print("✅ Watermark applied.")
print(f"   Column    : order_date")
print(f"   Threshold : 10 minutes")
print(f"   isStreaming : {stream_wm.isStreaming}")   # True

In [0]:
print("▶ Starting stream — First Run...")
print("-" * 40)

live_query_1 = (
    stream_wm.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_LIVE)
        .trigger(availableNow=True)
        .table(f"{SCHEMA_G}.live_orders")
)
live_query_1.awaitTermination()

# Check for exceptions
exc = live_query_1.exception()
if exc:
    print(f"❌ Stream failed: {exc}")
else:
    print(f"✅ Stream completed successfully.")

print("-" * 40)

In [0]:
# Verify table exists
try:
    count_before = spark.read.table(f"{SCHEMA_G}.live_orders").count()
    print(f"✅ Table exists : {SCHEMA_G}.live_orders")
    print(f"   Rows written : {count_before}")   # 235
except Exception as e:
    print(f"❌ Table not found: {e}")
    print("   Re-run Cell 40 to Cell 46 again.")

# Verify checkpoint was populated
print("\n=== Checkpoint contents ===")
try:
    ck_files = dbutils.fs.ls(CHECKPOINT_LIVE)
    for f in ck_files:
        print(f"  - {f.name}")
    print(f"✅ Checkpoint populated at : {CHECKPOINT_LIVE}")
except Exception as e:
    print(f"❌ Checkpoint empty: {e}")

In [0]:
print("=== First 10 rows ===")
spark.read.table(f"{SCHEMA_G}.live_orders") \
     .orderBy("order_id") \
     .show(10, truncate=False)

print("=== Row count by region ===")
spark.sql(f"""
    SELECT region, COUNT(*) AS row_count
    FROM   {SCHEMA_G}.live_orders
    GROUP BY region
    ORDER BY region
""").show()

print("=== Batch2 rows only (ORD0201+) ===")
batch2_count = (
    spark.read.table(f"{SCHEMA_G}.live_orders")
         .filter(col("order_id") >= "ORD0201")
         .count()
)
print(f"   Batch2 rows : {batch2_count}")   # 30

In [0]:
active = spark.streams.active
print(f"Active streams : {len(active)}")

if len(active) == 0:
    print("✅ Stream fully stopped.")
else:
    for s in active:
        s.stop()
        print(f"   Force stopped : {s.name}")
    print("✅ All streams stopped.")

In [0]:
print("▶ Restarting stream — same checkpoint...")
print(f"   Checkpoint : {CHECKPOINT_LIVE}")
print("-" * 40)

live_query_2 = (
    stream_wm.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_LIVE)   # same path
        .trigger(availableNow=True)
        .table(f"{SCHEMA_G}.live_orders")
)
live_query_2.awaitTermination()

# Check for exceptions
exc = live_query_2.exception()
if exc:
    print(f"❌ Stream failed: {exc}")
else:
    print(f"✅ Stream restarted successfully.")

count_after = spark.read.table(f"{SCHEMA_G}.live_orders").count()
print("-" * 40)
print(f"   Rows after restart : {count_after}")

In [0]:
print("=" * 50)
print("IDEMPOTENCY CHECK")
print("=" * 50)
print(f"  Row count before restart : {count_before}")
print(f"  Row count after  restart : {count_after}")
print("-" * 50)

if count_before == count_after:
    print(f"  ✅ PASSED — No duplicate rows written.")
    print(f"     Checkpoint correctly tracked all processed files.")
    print(f"     On restart, Spark skipped already processed files.")
else:
    diff = count_after - count_before
    print(f"  ❌ FAILED — {diff} extra rows found!")
    print(f"     Ensure checkpointLocation is identical on both runs.")

In [0]:
print("=" * 50)
print("TASK 5 — STRUCTURED STREAMING SUMMARY")
print("=" * 50)
print(f"""
  Source path      : {ORDERS_RAW_PATH}
  Checkpoint path  : {CHECKPOINT_LIVE}
  Schema location  : {SCHEMA_LOC_LIVE}
  Target table     : {SCHEMA_G}.live_orders

  Source files     :
     orders.csv        → 205 rows
     orders_batch2.csv → 30  rows

  Watermark        : order_date — 10 minutes
  Output mode      : append
  Trigger          : availableNow

  Row counts       :
     After 1st run  → {count_before}
     After restart  → {count_after}

  Idempotency : {"✅ Verified — No duplicates" if count_before == count_after else "❌ Failed — Duplicates found"}
""")
print("✅ Task 5 complete. Proceed to Task 6 — Workflow Orchestration.")